# NPDES Data Cleaning: Discharge Monitoring Reports (Iowa, FY2015–FY2025)

Cleans EPA ECHO's **Discharge Monitoring Report (DMR)** effluent-chart exports
for Iowa into one tidy long table: one row per reported limit/measurement record.
Each row pairs a permit-feature parameter's effluent **limit** with the
facility's **reported value** for a monitoring period.

**Inputs:**  `data/tabular/01_raw/npdes/NPDES_DMRS_FY{2015..2025}.csv`
  (eleven fiscal-year files, each already scoped to Iowa permits)
**Output:** `data/tabular/02_clean/npdes/npdes-dmrs-clean.csv`

The raw export is 57 columns wide and very granular. We keep the fields needed
for water-quality analysis — who/where (permit, outfall), what (parameter,
monitoring location, statistical basis), the permitted limit, the reported value
in standard units, data-quality qualifiers, and compliance (exceedance %,
violation, lateness) — and standardize types.

**Cleaning steps** — read the eleven fiscal years (only the kept columns),
stamp each with its `fiscal_year`, assert Iowa scope, rename to snake_case,
parse dates, coerce numeric values, then de-duplicate and write.

In [1]:
import re
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "npdes"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "npdes"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/npdes


## Step 1 — Define the column contract

`KEEP` maps the raw columns we retain to their clean snake_case names. Reading
only these columns (via `usecols`) keeps the eleven-file load fast and the memory
footprint small. `LIMIT_VALUE_STANDARD_UNITS` / `DMR_VALUE_STANDARD_UNITS` are
the limit and reported value already converted to a common unit
(`STANDARD_UNIT_DESC`), which is what makes values comparable across permits.

In [2]:
KEEP = {
    "EXTERNAL_PERMIT_NMBR": "npdes_id",
    "VERSION_NMBR": "version_nmbr",
    "PERM_FEATURE_NMBR": "perm_feature_nmbr",
    "PERM_FEATURE_TYPE_CODE": "perm_feature_type_code",
    "LIMIT_SET_DESIGNATOR": "limit_set_designator",
    "PARAMETER_CODE": "parameter_code",
    "PARAMETER_DESC": "parameter_desc",
    "MONITORING_LOCATION_CODE": "monitoring_location_code",
    "LIMIT_BEGIN_DATE": "limit_begin_date",
    "LIMIT_END_DATE": "limit_end_date",
    "STATISTICAL_BASE_CODE": "statistical_base_code",
    "LIMIT_VALUE_TYPE_CODE": "limit_value_type_code",
    "LIMIT_VALUE_STANDARD_UNITS": "limit_value",
    "LIMIT_VALUE_QUALIFIER_CODE": "limit_value_qualifier",
    "STANDARD_UNIT_DESC": "standard_unit_desc",
    "MONITORING_PERIOD_END_DATE": "monitoring_period_end_date",
    "VALUE_TYPE_CODE": "value_type_code",
    "DMR_VALUE_STANDARD_UNITS": "dmr_value",
    "DMR_VALUE_QUALIFIER_CODE": "dmr_value_qualifier",
    "DMR_FORM_VALUE_ID": "dmr_form_value_id",
    "VALUE_RECEIVED_DATE": "value_received_date",
    "NODI_CODE": "nodi_code",
    "EXCEEDENCE_PCT": "exceedence_pct",
    "VIOLATION_CODE": "violation_code",
    "DAYS_LATE": "days_late",
}
DATE_COLS = ["limit_begin_date", "limit_end_date",
             "monitoring_period_end_date", "value_received_date"]
NUM_COLS = ["limit_value", "dmr_value", "exceedence_pct", "days_late"]
print(f"Keeping {len(KEEP)} of 57 columns")

Keeping 25 of 57 columns


## Step 2 — Load and concatenate the eleven fiscal years

Each file is read as strings (preserving codes), tagged with the `fiscal_year`
parsed from its filename, and stacked. Every file is an Iowa-only export, which
we assert before concatenating.

In [3]:
files = sorted(RAW_DIR.glob("NPDES_DMRS_FY*.csv"))
assert files, "no NPDES_DMRS_FY*.csv files found"

frames = []
for f in files:
    fy = int(re.search(r"FY(\d{4})", f.name).group(1))
    part = pd.read_csv(f, usecols=list(KEEP), dtype="string").rename(columns=KEEP)
    assert part["npdes_id"].str.startswith("IA").all(), f"non-Iowa permit in {f.name}"
    part["fiscal_year"] = fy
    frames.append(part)
    print(f"  {f.name}: {len(part):,} rows")

df = pd.concat(frames, ignore_index=True)
n_raw = len(df)
print(f"\nConcatenated {n_raw:,} rows across {len(files)} fiscal years, "
      f"{df['npdes_id'].nunique()} permits")

  NPDES_DMRS_FY2015.csv: 156,218 rows


  NPDES_DMRS_FY2016.csv: 173,082 rows


  NPDES_DMRS_FY2017.csv: 257,280 rows


  NPDES_DMRS_FY2018.csv: 254,509 rows


  NPDES_DMRS_FY2019.csv: 228,367 rows


...

  NPDES_DMRS_FY2022.csv: 299,953 rows


  NPDES_DMRS_FY2023.csv: 308,609 rows


  NPDES_DMRS_FY2024.csv: 294,020 rows


  NPDES_DMRS_FY2025.csv: 291,232 rows



Concatenated 2,786,182 rows across 11 fiscal years, 1469 permits


## Step 3 — Parse dates

Dates arrive as `MM/DD/YYYY`; parse them to real dates. `monitoring_period_end_date`
is the analytical timestamp (when the reported value applies) and must always be
present.

In [4]:
for c in DATE_COLS:
    df[c] = pd.to_datetime(df[c], format="%m/%d/%Y", errors="coerce").dt.date
assert df["monitoring_period_end_date"].notna().all(), "missing monitoring period end date"
mped = pd.to_datetime(df["monitoring_period_end_date"])
print("Monitoring period range:", mped.min().date(), "→", mped.max().date())

Monitoring period range: 2014-10-31 → 2025-09-30


## Step 4 — Coerce numeric values

The limit, reported value, exceedance percentage, and days-late are quantities.
`version_nmbr` is a small integer. A null `dmr_value` is a genuine non-report
(see `nodi_code` for why) and is preserved as missing, not zero.

In [5]:
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["version_nmbr"] = pd.to_numeric(df["version_nmbr"], errors="coerce").astype("Int64")

print(f"Rows with a reported value: {df['dmr_value'].notna().sum():,} / {len(df):,}")
print(f"Rows with a no-data indicator (nodi_code): {df['nodi_code'].notna().sum():,}")
print("\ndmr_value describe:")
print(df["dmr_value"].describe().to_string())

Rows with a reported value: 1,739,702 / 2,786,182
Rows with a no-data indicator (nodi_code): 955,974

dmr_value describe:
count       1739702.0
mean       322.395555
std       17176.97932
min     -18648.776243
25%          3.565994
50%               8.3
75%              50.7
max        11000000.0


## Step 5 — De-duplicate and order columns

Drop exact duplicate rows, then order columns logically (keys → parameter →
limit → reported value → compliance) and sort by permit, outfall, parameter, and
monitoring period.

In [6]:
df = df.drop_duplicates().reset_index(drop=True)

ORDERED = [
    "npdes_id", "fiscal_year", "perm_feature_nmbr", "perm_feature_type_code",
    "limit_set_designator", "parameter_code", "parameter_desc",
    "monitoring_location_code", "statistical_base_code",
    "monitoring_period_end_date", "limit_begin_date", "limit_end_date",
    "limit_value_type_code", "limit_value", "limit_value_qualifier",
    "value_type_code", "dmr_value", "dmr_value_qualifier", "standard_unit_desc",
    "nodi_code", "exceedence_pct", "violation_code", "days_late",
    "version_nmbr", "value_received_date", "dmr_form_value_id",
]
assert set(ORDERED) == set(df.columns), set(df.columns) ^ set(ORDERED)
df = df[ORDERED].sort_values(
    ["npdes_id", "perm_feature_nmbr", "parameter_code", "monitoring_period_end_date"]
).reset_index(drop=True)
print(f"Rows after de-duplication: {len(df):,} (from {n_raw:,})")

Rows after de-duplication: 2,786,182 (from 2,786,182)


## Step 6 — Sanity check and write

A quick look at the most-reported parameters confirms the table is sensible
(suspended solids, pH, BOD, nutrients), then write the tidy table.

In [7]:
print("Top reported parameters:")
print(df["parameter_desc"].value_counts().head(10).to_string())

out_path = CLEAN_DIR / "npdes-dmrs-clean.csv"
df.to_csv(out_path, index=False)
print(f"\nWrote {len(df):,} rows × {df.shape[1]} cols to:")
print(" ", out_path.relative_to(REPO_ROOT))
df.head()

Top reported parameters:
parameter_desc
Solids, total suspended             637040
pH                                  462125
BOD, carbonaceous [5 day, 20 C]     389800
Nitrogen, ammonia total [as N]      229823
BOD, 5-day, 20 deg. C               189156
Nitrogen, total [as N]              100961
Chlorine, total residual             98172
Phosphorus, total [as P]             93365
Nitrogen, Kjeldahl, total [as N]     76297
Chloride [as Cl]                     51487



Wrote 2,786,182 rows × 26 cols to:
  data/tabular/02_clean/npdes/npdes-dmrs-clean.csv


,npdes_id,fiscal_year,perm_feature_nmbr,perm_feature_type_code,limit_set_designator,parameter_code,parameter_desc,monitoring_location_code,statistical_base_code,monitoring_period_end_date,...,dmr_value,dmr_value_qualifier,standard_unit_desc,nodi_code,exceedence_pct,violation_code,days_late,version_nmbr,value_received_date,dmr_form_value_id
0,IA0000035,2017,001,EXO,A,00400,pH,1,DD,2017-04-30,...,10.79,=,<NA>,<NA>,<NA>,<NA>,<NA>,3,2017-05-15,3646134596
1,IA0000035,2017,001,EXO,A,00400,pH,1,DC,2017-04-30,...,10.79,=,<NA>,<NA>,<NA>,<NA>,<NA>,3,2017-05-15,3646134597
2,IA0000035,2018,001,EXO,A,00400,pH,1,DD,2018-04-30,...,10.08,=,SU,<NA>,<NA>,<NA>,<NA>,3,2018-05-09,3646134598
3,IA0000035,2018,001,EXO,A,00400,pH,1,DC,2018-04-30,...,10.08,=,SU,<NA>,<NA>,<NA>,<NA>,3,2018-05-09,3646134599
4,IA0000035,2019,001,EXO,A,00400,pH,1,DC,2019-04-30,...,<NA>,<NA>,SU,<NA>,<NA>,D90,<NA>,3,NaT,3646134601
